# 09 — LOGO evaluation for baseline CNN and frequency models

Evaluate our three models when one AI generator is completely unseen during training and validation. This notebook deliberately reuses the exact LOGO manifest built for ResNet18 and ViT, making all five model families directly comparable.

Run the notebook once per generator by changing `UNSEEN_GENERATORS` in the setup cell. Do not run multiple folds in parallel because each CNN training needs its own GPU/CPU resources.


## 1. Setup

The standard project hyperparameters remain fixed. Only the held-out generator changes between runs. Checkpoints and fitted frequency models are written to ignored folders; metrics, predictions, and figures are retained as experiment evidence.


In [1]:
import json
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.optim import AdamW


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "experiments.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not find the repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT))

from src.data.config import load_config
from src.data.loaders import create_dataloaders
from src.data.logo import build_logo_manifest, list_ai_generators
from src.evaluation import (
    evaluate_predictions,
    evaluate_probability_predictions,
    save_evaluation_outputs,
)
from src.features import (
    compute_log_power_grid,
    compute_normalized_power_spectrum,
    compute_radial_energy_profile,
    image_to_luminance,
)
from src.models import BaselineCNN, count_trainable_parameters
from src.training import fit, set_seed

config = load_config()
SEED = config["seed"]
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Change only this list before starting the next LOGO fold.
UNSEEN_GENERATORS = ["sdv1_5", "wukong"]
UNSEEN_LABEL = "+".join(UNSEEN_GENERATORS)

MANIFEST_PATH = REPO_ROOT / "data" / "dataset.csv"
PROCESSED_ROOT = REPO_ROOT / "data" / "processed"
CHECKPOINT_ROOT = REPO_ROOT / "results" / "checkpoints" / f"logo_{UNSEEN_LABEL}"
MODEL_ROOT = REPO_ROOT / "results" / "models" / f"logo_{UNSEEN_LABEL}"
METRICS_ROOT = REPO_ROOT / "results" / "metrics" / f"logo_{UNSEEN_LABEL}"
FIGURES_ROOT = REPO_ROOT / "results" / "figures" / f"logo_{UNSEEN_LABEL}"

for path in (CHECKPOINT_ROOT, MODEL_ROOT, METRICS_ROOT, FIGURES_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print(f"Unseen generator(s): {UNSEEN_GENERATORS}")

Device: cuda
GPU: NVIDIA GeForce GTX 1660 Ti
Unseen generator(s): ['sdv1_5', 'wukong']


## 2. Build and verify the shared LOGO split

All AI images made by the held-out generator are moved to test. AI test images from seen generators are excluded. Real images retain the same splits used by the colleague's ResNet18/ViT experiment.

The assertions are essential: they prove that the held-out AI generator cannot influence training, validation, early stopping, or Logistic Regression model selection.


In [2]:
manifest = pd.read_csv(MANIFEST_PATH)
all_generators = list_ai_generators(manifest)
assert all(g in all_generators for g in UNSEEN_GENERATORS)

logo_manifest = build_logo_manifest(manifest, UNSEEN_GENERATORS)

train_ai_generators = sorted(logo_manifest.loc[
    (logo_manifest["split"] == "train") & (logo_manifest["label"] == "ai"), "generator"
].unique())
val_ai_generators = sorted(logo_manifest.loc[
    (logo_manifest["split"] == "val") & (logo_manifest["label"] == "ai"), "generator"
].unique())
test_ai_generators = sorted(logo_manifest.loc[
    (logo_manifest["split"] == "test") & (logo_manifest["label"] == "ai"), "generator"
].unique())

assert not (set(UNSEEN_GENERATORS) & set(train_ai_generators))
assert not (set(UNSEEN_GENERATORS) & set(val_ai_generators))
assert set(test_ai_generators) == set(UNSEEN_GENERATORS)

display(logo_manifest.groupby(["split", "label"]).size().rename("images"))
print("Train AI generators:", train_ai_generators)
print("Val AI generators:  ", val_ai_generators)
print("Test AI generators: ", test_ai_generators)
print("LOGO leakage checks passed.")


split  label
test   ai       2000
       real     1050
train  ai       3500
       real     4900
val    ai        750
       real     1050
Name: images, dtype: int64

Train AI generators: ['adm', 'biggan', 'glide', 'midjourney', 'vqdm']
Val AI generators:   ['adm', 'biggan', 'glide', 'midjourney', 'vqdm']
Test AI generators:  ['sdv1_5', 'wukong']
LOGO leakage checks passed.


## 3. Baseline CNN

This is the same four-block CNN, normalization, optimizer, and early-stopping configuration used in notebook 03. It is trained from scratch for this fold. Validation loss chooses the best epoch; test remains untouched until that choice is complete.


In [3]:
loader_config = config["dataloader"]
cnn_loaders = create_dataloaders(
    manifest=logo_manifest,
    processed_root=PROCESSED_ROOT,
    batch_size=loader_config["batch_size"],
    num_workers=loader_config["num_workers"],
    seed=SEED,
    pin_memory=device.type == "cuda",
)

model_config = config["baseline_cnn"]
cnn_model = BaselineCNN(
    channels=tuple(model_config["channels"]),
    dropout=model_config["dropout"],
).to(device)
cnn_criterion = nn.BCEWithLogitsLoss()
cnn_optimizer = AdamW(
    cnn_model.parameters(),
    lr=model_config["learning_rate"],
    weight_decay=model_config["weight_decay"],
)

print("Dataset sizes:", {split: len(loader.dataset) for split, loader in cnn_loaders.items()})
print(f"Trainable parameters: {count_trainable_parameters(cnn_model):,}")


Dataset sizes: {'train': 8400, 'val': 1800, 'test': 3050}
Trainable parameters: 389,153


In [4]:
# This is the long-running CNN training cell.
cnn_fit_result = fit(
    model=cnn_model,
    train_dataloader=cnn_loaders["train"],
    val_dataloader=cnn_loaders["val"],
    criterion=cnn_criterion,
    optimizer=cnn_optimizer,
    device=device,
    epochs=model_config["epochs"],
    patience=model_config["early_stopping_patience"],
    min_delta=model_config["early_stopping_min_delta"],
    checkpoint_path=CHECKPOINT_ROOT / "baseline_cnn.pt",
)
print(
    f"Best epoch: {cnn_fit_result.best_epoch}; "
    f"best validation loss: {cnn_fit_result.best_val_loss:.4f}"
)


Epoch 01/20 | train loss 0.5601, F1 0.6642 | val loss 0.4893, F1 0.6942
Epoch 02/20 | train loss 0.4862, F1 0.7274 | val loss 0.5023, F1 0.7144
Epoch 03/20 | train loss 0.4741, F1 0.7391 | val loss 0.4506, F1 0.7574
Epoch 04/20 | train loss 0.4652, F1 0.7407 | val loss 0.4459, F1 0.7618
Epoch 05/20 | train loss 0.4557, F1 0.7445 | val loss 0.4375, F1 0.7572
Epoch 06/20 | train loss 0.4456, F1 0.7493 | val loss 0.4602, F1 0.7502
Epoch 07/20 | train loss 0.4381, F1 0.7562 | val loss 0.4714, F1 0.7150
Epoch 08/20 | train loss 0.4342, F1 0.7664 | val loss 0.4343, F1 0.7593
Epoch 09/20 | train loss 0.4394, F1 0.7565 | val loss 0.4310, F1 0.7589
Epoch 10/20 | train loss 0.4331, F1 0.7596 | val loss 0.4455, F1 0.7598
Epoch 11/20 | train loss 0.4251, F1 0.7659 | val loss 0.4545, F1 0.7509
Epoch 12/20 | train loss 0.4204, F1 0.7683 | val loss 0.4122, F1 0.7699
Epoch 13/20 | train loss 0.4216, F1 0.7697 | val loss 0.4154, F1 0.7728
Epoch 14/20 | train loss 0.4135, F1 0.7739 | val loss 0.4100, F1

In [5]:
cnn_test_result = evaluate_predictions(
    model=cnn_model,
    dataloader=cnn_loaders["test"],
    criterion=cnn_criterion,
    device=device,
    threshold=0.5,
)
display(pd.Series(cnn_test_result.metrics.to_dict(), name="baseline_cnn"))

cnn_saved_paths = save_evaluation_outputs(
    cnn_test_result,
    output_dir=METRICS_ROOT / "baseline_cnn",
    figures_dir=FIGURES_ROOT / "baseline_cnn",
    model_label="Baseline CNN",
)
print("CNN outputs saved.")


loss         1.406047
accuracy     0.357705
precision    0.587234
recall       0.069000
f1           0.123490
roc_auc      0.566502
Name: baseline_cnn, dtype: float64

CNN outputs saved.


## 4. Frequency representations

Each image is transformed to luminance, then to a normalized FFT power spectrum. The radial model uses 64 annular energy bands. The hybrid model combines those 64 values with a 16×16 log-power grid, for 320 features.

Feature extraction is deterministic and contains no learning. The scaler and Logistic Regression are fitted only on train. Candidate regularization strengths are compared only on validation loss.


In [6]:
RADIAL_BINS = 64
GRID_SIZE = 16
THRESHOLD = 0.5


def extract_frequency_features(rows: pd.DataFrame):
    radial = np.empty((len(rows), RADIAL_BINS), dtype=np.float32)
    grid = np.empty((len(rows), GRID_SIZE * GRID_SIZE), dtype=np.float32)

    for index, filename in enumerate(rows["filename"]):
        with Image.open(PROCESSED_ROOT / filename) as image:
            luminance = image_to_luminance(image)
        power = compute_normalized_power_spectrum(luminance)
        radial[index] = compute_radial_energy_profile(power, bins=RADIAL_BINS)
        grid[index] = compute_log_power_grid(power, grid_size=GRID_SIZE).ravel()

        if (index + 1) % 1000 == 0 or index + 1 == len(rows):
            print(f"  extracted {index + 1:,}/{len(rows):,}")

    hybrid = np.concatenate((radial, grid), axis=1)
    labels = rows["label"].map({"real": 0, "ai": 1}).to_numpy(dtype=np.int64)
    return radial, hybrid, labels


def build_frequency_model(c_value: float) -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(C=c_value, max_iter=3000, random_state=SEED)),
    ])


def select_regularization(X_train, y_train, X_val, y_val, candidates):
    rows = []
    fitted = {}
    for c_value in candidates:
        candidate = build_frequency_model(c_value)
        candidate.fit(X_train, y_train)
        probabilities = candidate.predict_proba(X_val)[:, 1]
        result = evaluate_probability_predictions(
            logo_manifest[logo_manifest["split"] == "val"].reset_index(drop=True),
            probabilities,
            threshold=THRESHOLD,
        )
        rows.append({"C": c_value, **result.metrics.to_dict()})
        fitted[c_value] = candidate

    table = pd.DataFrame(rows).sort_values("loss").reset_index(drop=True)
    best_c = float(table.iloc[0]["C"])
    return fitted[best_c], best_c, table


In [7]:
# This is the long-running FFT extraction cell.
train_frame = logo_manifest[logo_manifest["split"] == "train"].reset_index(drop=True)
val_frame = logo_manifest[logo_manifest["split"] == "val"].reset_index(drop=True)
test_frame = logo_manifest[logo_manifest["split"] == "test"].reset_index(drop=True)

print("Train:")
X_train_radial, X_train_hybrid, y_train = extract_frequency_features(train_frame)
print("Validation:")
X_val_radial, X_val_hybrid, y_val = extract_frequency_features(val_frame)
print("Test:")
X_test_radial, X_test_hybrid, y_test = extract_frequency_features(test_frame)

print("Radial:", X_train_radial.shape, X_val_radial.shape, X_test_radial.shape)
print("Hybrid:", X_train_hybrid.shape, X_val_hybrid.shape, X_test_hybrid.shape)


Train:
  extracted 1,000/8,400
  extracted 2,000/8,400
  extracted 3,000/8,400
  extracted 4,000/8,400
  extracted 5,000/8,400
  extracted 6,000/8,400
  extracted 7,000/8,400
  extracted 8,000/8,400
  extracted 8,400/8,400
Validation:
  extracted 1,000/1,800
  extracted 1,800/1,800
Test:
  extracted 1,000/3,050
  extracted 2,000/3,050
  extracted 3,000/3,050
  extracted 3,050/3,050
Radial: (8400, 64) (1800, 64) (3050, 64)
Hybrid: (8400, 320) (1800, 320) (3050, 320)


In [8]:
radial_model, radial_best_c, radial_selection = select_regularization(
    X_train_radial, y_train, X_val_radial, y_val,
    candidates=[0.01, 0.1, 1.0, 10.0],
)
hybrid_model, hybrid_best_c, hybrid_selection = select_regularization(
    X_train_hybrid, y_train, X_val_hybrid, y_val,
    candidates=[0.001, 0.01, 0.1, 1.0],
)

print(f"Selected radial C: {radial_best_c}")
display(radial_selection)
print(f"Selected hybrid C: {hybrid_best_c}")
display(hybrid_selection)


Selected radial C: 0.01


,C,loss,accuracy,precision,recall,f1,roc_auc
0,0.01,0.621217,0.721111,0.698083,0.582667,0.635174,0.766741
1,0.10,0.622817,0.717222,0.685670,0.593333,0.636169,0.765472
2,1.00,0.624370,0.717778,0.682229,0.604000,0.640736,0.764085
3,10.00,0.624634,0.717778,0.680060,0.609333,0.642757,0.763834


Selected hybrid C: 0.01


,C,loss,accuracy,precision,recall,f1,roc_auc
0,0.010,0.508085,0.756667,0.778571,0.581333,0.665649,0.816399
1,0.100,0.511345,0.753333,0.759322,0.597333,0.668657,0.813528
2,0.001,0.515197,0.753333,0.776173,0.573333,0.659509,0.812391
3,1.000,0.517339,0.750556,0.748760,0.604000,0.668635,0.809150


In [9]:
frequency_results = {}
frequency_models = {
    "frequency_radial": (radial_model, X_test_radial),
    "frequency_hybrid": (hybrid_model, X_test_hybrid),
}

for model_name, (fitted_model, X_test) in frequency_models.items():
    probabilities = fitted_model.predict_proba(X_test)[:, 1]
    result = evaluate_probability_predictions(test_frame, probabilities, threshold=THRESHOLD)
    frequency_results[model_name] = result

    display(pd.Series(result.metrics.to_dict(), name=model_name))
    save_evaluation_outputs(
        result,
        output_dir=METRICS_ROOT / model_name,
        figures_dir=FIGURES_ROOT / model_name,
        model_label=model_name.replace("_", " ").title(),
    )
    joblib.dump(fitted_model, MODEL_ROOT / f"{model_name}.joblib")

radial_selection.to_csv(METRICS_ROOT / "frequency_radial" / "validation_selection.csv", index=False)
hybrid_selection.to_csv(METRICS_ROOT / "frequency_hybrid" / "validation_selection.csv", index=False)
print("Frequency outputs saved.")


loss         0.932848
accuracy     0.403934
precision    0.672348
recall       0.177500
f1           0.280854
roc_auc      0.547508
Name: frequency_radial, dtype: float64

loss         0.980122
accuracy     0.374098
precision    0.663082
recall       0.092500
f1           0.162352
roc_auc      0.611827
Name: frequency_hybrid, dtype: float64

Frequency outputs saved.


## 5. Preserve and extend the shared comparison

The existing per-fold CSV may already contain the colleague's ResNet18 and ViT rows. This cell performs an upsert: it replaces only rows for our three models and preserves every other model. It also rebuilds the cross-fold `logo_summary.csv`.


In [10]:
METRIC_NAMES = ["accuracy", "precision", "recall", "f1", "roc_auc"]
our_results = {
    "baseline_cnn": cnn_test_result.metrics.to_dict(),
    "frequency_radial": frequency_results["frequency_radial"].metrics.to_dict(),
    "frequency_hybrid": frequency_results["frequency_hybrid"].metrics.to_dict(),
}
seen_metric_paths = {
    "baseline_cnn": REPO_ROOT / "results" / "metrics" / "baseline_cnn" / "test_metrics.json",
    "frequency_radial": REPO_ROOT / "results" / "metrics" / "frequency" / "radial" / "test_metrics.json",
    "frequency_hybrid": REPO_ROOT / "results" / "metrics" / "frequency" / "hybrid" / "test_metrics.json",
}

new_rows = []
for model_name, unseen_metrics in our_results.items():
    new_rows.append({
        "model": model_name,
        "split": "unseen",
        **{metric: unseen_metrics[metric] for metric in METRIC_NAMES},
    })
    seen_path = seen_metric_paths[model_name]
    if seen_path.is_file():
        seen_metrics = json.loads(seen_path.read_text(encoding="utf-8"))
        new_rows.append({
            "model": model_name,
            "split": "seen",
            **{metric: seen_metrics[metric] for metric in METRIC_NAMES},
        })
    else:
        print(f"Missing standard metrics: {seen_path.relative_to(REPO_ROOT)}")

comparison_path = METRICS_ROOT / "seen_vs_unseen_comparison.csv"
new_comparison = pd.DataFrame(new_rows)
if comparison_path.is_file():
    existing_comparison = pd.read_csv(comparison_path)
    existing_comparison = existing_comparison[
        ~existing_comparison["model"].isin(our_results)
    ]
    comparison = pd.concat([existing_comparison, new_comparison], ignore_index=True)
else:
    comparison = new_comparison

comparison = comparison.sort_values(["model", "split"]).reset_index(drop=True)
comparison.to_csv(comparison_path, index=False)
display(comparison)

summary_frames = []
for path in sorted((REPO_ROOT / "results" / "metrics").glob("logo_*/seen_vs_unseen_comparison.csv")):
    per_run = pd.read_csv(path)
    per_run.insert(0, "unseen_generators", path.parent.name.removeprefix("logo_"))
    summary_frames.append(per_run)

logo_summary = pd.concat(summary_frames, ignore_index=True).sort_values(
    ["unseen_generators", "model", "split"]
)
summary_path = REPO_ROOT / "results" / "metrics" / "logo_summary.csv"
logo_summary.to_csv(summary_path, index=False)
print(f"Saved fold comparison: {comparison_path.relative_to(REPO_ROOT)}")
print(f"Updated summary: {summary_path.relative_to(REPO_ROOT)}")


,model,split,accuracy,precision,recall,f1,roc_auc
0,baseline_cnn,seen,0.796190,0.789033,0.808571,0.798683,0.887641
1,baseline_cnn,unseen,0.357705,0.587234,0.069000,0.123490,0.566502
2,frequency_hybrid,seen,0.708095,0.754366,0.617143,0.678889,0.794697
3,frequency_hybrid,unseen,0.374098,0.663082,0.092500,0.162352,0.611827
4,frequency_radial,seen,0.649048,0.618471,0.778095,0.689161,0.704762
5,frequency_radial,unseen,0.403934,0.672348,0.177500,0.280854,0.547508
6,resnet18,seen,0.819048,0.799107,0.852381,0.824885,0.906151
7,resnet18,unseen,0.541311,0.857313,0.360500,0.507568,0.726903
8,vit_small,seen,0.890000,0.880930,0.901905,0.891294,0.956613
9,vit_small,unseen,0.546230,0.901042,0.346000,0.500000,0.788165


Saved fold comparison: results\metrics\logo_sdv1_5+wukong\seen_vs_unseen_comparison.csv
Updated summary: results\metrics\logo_summary.csv


## How to run the next fold

After one fold finishes and all outputs are saved:

1. Save the notebook.
2. Restart the kernel to release CNN and feature arrays.
3. Change only `UNSEEN_GENERATORS` in the setup cell.
4. Run all cells from the top again.

Standard seven-fold LOGO uses: `adm`, `biggan`, `glide`, `midjourney`, `sdv1_5`, `vqdm`, and `wukong`. The optional two-generator `sdv1_5+wukong` experiment can be repeated later, after all single-generator folds.


## LOGO Results Across Models

All seven single-generator LOGO folds and the joint `sdv1_5+wukong` holdout
have been evaluated for all five model families: `baseline_cnn`,
`frequency_radial`, `frequency_hybrid`, `resnet18`, and `vit_small`.

The table below reports the complete set of evaluation metrics for each
unseen-generator fold. The corresponding seen-generator results are included
as a reference baseline. The machine-readable source of truth is
`results/metrics/logo_summary.csv`.

| Unseen | Model | accuracy | precision | recall | f1 | roc_auc |
|---|---|---|---|---|---|---|
| -- | baseline_cnn (seen) | 0.796 | 0.789 | 0.809 | 0.799 | 0.888 |
| -- | frequency_radial (seen) | 0.649 | 0.618 | 0.778 | 0.689 | 0.705 |
| -- | frequency_hybrid (seen) | 0.708 | 0.754 | 0.617 | 0.679 | 0.795 |
| -- | resnet18 (seen) | 0.819 | 0.799 | 0.852 | 0.825 | 0.906 |
| -- | vit_small (seen) | 0.890 | 0.881 | 0.902 | 0.891 | 0.957 |
| biggan | baseline_cnn | 0.817 | 0.822 | 0.796 | 0.809 | 0.901 |
| biggan | frequency_radial | 0.750 | 0.753 | 0.727 | 0.740 | 0.851 |
| biggan | **frequency_hybrid** | 0.899 | 0.832 | **0.992** | **0.905** | **0.970** |
| biggan | resnet18 | 0.813 | 0.807 | 0.810 | 0.808 | 0.887 |
| biggan | vit_small | 0.871 | 0.883 | 0.849 | 0.865 | 0.949 |
| glide | baseline_cnn | 0.864 | 0.855 | 0.870 | 0.862 | 0.936 |
| glide | frequency_radial | 0.693 | 0.722 | 0.604 | 0.658 | 0.805 |
| glide | **frequency_hybrid** | 0.898 | 0.838 | **0.979** | **0.903** | **0.967** |
| glide | resnet18 | 0.831 | 0.811 | 0.852 | 0.831 | 0.909 |
| glide | vit_small | 0.906 | 0.907 | 0.899 | 0.903 | 0.963 |
| adm | baseline_cnn | 0.601 | 0.644 | 0.410 | 0.501 | 0.667 |
| adm | frequency_radial | 0.611 | 0.649 | 0.441 | 0.525 | 0.690 |
| adm | **frequency_hybrid** | 0.653 | 0.723 | **0.467** | **0.567** | **0.730** |
| adm | resnet18 | 0.616 | 0.686 | 0.393 | 0.500 | 0.692 |
| adm | vit_small | 0.601 | 0.770 | 0.261 | 0.390 | 0.709 |
| vqdm | baseline_cnn | 0.573 | 0.651 | 0.270 | 0.382 | 0.648 |
| vqdm | frequency_radial | 0.504 | 0.484 | 0.258 | 0.337 | 0.518 |
| vqdm | frequency_hybrid | 0.580 | 0.649 | 0.301 | 0.411 | 0.613 |
| vqdm | resnet18 | 0.602 | 0.718 | 0.305 | 0.428 | 0.683 |
| vqdm | vit_small | 0.608 | 0.769 | 0.280 | 0.411 | 0.732 |
| midjourney | baseline_cnn | 0.526 | 0.553 | 0.152 | 0.238 | 0.607 |
| midjourney | frequency_radial | 0.604 | 0.640 | 0.432 | 0.516 | 0.672 |
| midjourney | frequency_hybrid | 0.576 | 0.658 | 0.271 | 0.384 | 0.697 |
| midjourney | resnet18 | 0.636 | 0.750 | 0.380 | 0.504 | 0.739 |
| midjourney | **vit_small** | 0.690 | 0.837 | **0.452** | **0.587** | **0.833** |
| sdv1_5 (alone) | baseline_cnn | 0.658 | 0.736 | 0.466 | 0.571 | 0.783 |
| sdv1_5 (alone) | frequency_radial | 0.577 | 0.585 | 0.457 | 0.513 | 0.602 |
| sdv1_5 (alone) | frequency_hybrid | 0.561 | 0.628 | 0.247 | 0.355 | 0.667 |
| sdv1_5 (alone) | **resnet18** | 0.788 | 0.814 | **0.734** | **0.772** | 0.874 |
| sdv1_5 (alone) | vit_small | 0.819 | 0.894 | 0.714 | 0.794 | **0.921** |
| wukong (alone) | baseline_cnn | 0.628 | 0.716 | 0.393 | 0.507 | 0.718 |
| wukong (alone) | frequency_radial | 0.572 | 0.578 | 0.457 | 0.510 | 0.597 |
| wukong (alone) | frequency_hybrid | 0.566 | 0.635 | 0.258 | 0.367 | 0.678 |
| wukong (alone) | **resnet18** | 0.765 | 0.791 | **0.705** | **0.746** | 0.848 |
| wukong (alone) | vit_small | 0.778 | 0.869 | 0.641 | 0.738 | **0.878** |
| sdv1_5+wukong | baseline_cnn | 0.358 | 0.587 | 0.069 | 0.123 | 0.567 |
| sdv1_5+wukong | frequency_radial | 0.404 | 0.672 | 0.178 | 0.281 | 0.548 |
| sdv1_5+wukong | frequency_hybrid | 0.374 | 0.663 | 0.093 | 0.162 | 0.612 |
| sdv1_5+wukong | **resnet18** | 0.541 | 0.857 | **0.361** | **0.508** | 0.727 |
| sdv1_5+wukong | vit_small | 0.546 | 0.901 | 0.346 | 0.500 | **0.788** |

### Frequency-domain features generalize particularly well to BigGAN and GLIDE

`frequency_hybrid` achieves the strongest LOGO performance on both `biggan` and
`glide`. It reaches an F1 score of 0.905 on `biggan` and 0.903 on `glide`, with
ROC-AUC values of 0.970 and 0.967, respectively.

These results are consistent with the hypothesis that some generator-specific
artifacts are sufficiently stable in the frequency domain to generalize to an
unseen generator. In these two folds, the frequency-domain representation
outperforms both `resnet18` and `vit_small`.

Importantly, this does not by itself prove that the models detect a specific
known artifact such as GAN upsampling or a particular GLIDE signature. It
shows that the frequency-based representation transfers particularly well to
these unseen generators.

### Frequency features do not reproduce the lineage-dependent transfer

The pattern is markedly different for the `sdv1_5`/`wukong` pair. Although
`resnet18` and `vit_small` retain moderate recall when either generator is held
out individually, `frequency_hybrid` performs substantially worse.

When `sdv1_5` and `wukong` are held out together, the performance of all models
drops further. `resnet18` and `vit_small` retain the highest recall among the
five model families (0.361 and 0.346), whereas `frequency_hybrid` drops to
0.093.

This suggests that the generalization between `sdv1_5` and `wukong` observed
with the deep models is not captured effectively by the frequency-domain
features used here. The results are therefore consistent with the hypothesis
that this transfer depends on representations beyond the selected
frequency-domain statistics.

### ADM reveals a model-dependent generalization pattern

`adm` provides an important contrast to the results obtained with `resnet18`
and `vit_small`. `frequency_hybrid` achieves the best performance among the
five model families on the `adm` fold, with recall of 0.467 and ROC-AUC of
0.730.

This suggests that `adm` may contain frequency-domain characteristics that
remain detectable even when the generator itself is completely unseen.
However, the experiment does not establish which specific characteristics are
responsible for this behavior.

The result is important because it shows that generator generalization cannot
necessarily be characterized independently of the model representation. A
generator that appears difficult for one model family may still expose
generalizable information to another representation.

### Overall interpretation

The LOGO experiments show that generalization to unseen generators is strongly
dependent on both the generator and the representation used by the detector.

Three patterns emerge:

1. **Strong frequency-domain transfer:** `biggan` and `glide` are detected
   particularly well by `frequency_hybrid`, suggesting that their
   generalizable characteristics are captured effectively by the selected
   spectral features.

2. **Deep-model transfer between related generators:** `sdv1_5` and `wukong`
   show substantially better transfer with `resnet18` and `vit_small` than
   with the frequency-based models. Their joint holdout produces a pronounced
   performance drop, consistent with the possibility that the deep models
   exploit similarities between the two generators.

3. **Representation-dependent difficulty:** `midjourney`, `vqdm`, and `adm`
   do not behave uniformly across model families. In particular, `adm` is
   substantially better detected by `frequency_hybrid` than by either
   `resnet18` or `vit_small`.

Overall, these results suggest that unseen-generator generalization is not a
single phenomenon. Different model families may exploit different
generalizable properties of generated images, and evaluating only one
architecture can therefore lead to incomplete conclusions about whether a
generator is intrinsically difficult to detect.